In [1]:
import sys
import os
from pathlib import Path
project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "config.py").exists()
)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


In [2]:
import database as db
import config as cfg
import pandas as pd

sql_file_path = cfg.FUNCTIONS_CHECK_SQL_FILE_PATH

Database folder path: D:\Masai\CapstoneProject\Zepto-Data-AI-Platform\data_pipeline\database


In [3]:
connection = db.create_and_connect_to_database()

Database connected!


In [4]:
category_dataframe = pd.read_sql_query(
    "SELECT * FROM category_master",
    connection
)

availability_dataframe = pd.read_sql_query(
    "SELECT * FROM availability_master",
    connection
)

books_dataframe = pd.read_sql_query(
    "SELECT * FROM books",
    connection
)

In [5]:


all_joined_data_query = (
    "SELECT "
    "b.title, "
    "b.price_gbp, "
    "b.price_inr, "
    "b.rating, "
    "c.category, "
    "CASE "
    "WHEN a.availability = 1 THEN 'Yes' "
    "ELSE 'No' "
    "END AS available "
    "FROM books b "
    "JOIN category_master c "
    "ON b.category_id = c.id "
    "JOIN availability_master a "
    "ON b.availability_id = a.id "
    "ORDER BY c.category, b.rating DESC, b.title ASC"
)
# DISTINCT requirement
distinct_categories_query = "SELECT DISTINCT category " \
                            "FROM category_master " \
                            "ORDER BY category"


# Additional IN requirement
books_with_selected_ratings_query = "SELECT title, rating " \
                                    "FROM books " \
                                    "WHERE rating IN (4, 5) " \
                                    "ORDER BY rating DESC, title ASC"


# Queries used to validate individual tables
select_all_books_query = "SELECT * FROM books"

select_all_categories_query = "SELECT * FROM category_master"

select_all_availability_query = "SELECT * FROM availability_master"



In [6]:
# HELPER FUNCTIONS
# --------------------------------------------------

def add_id_to_dataframes(df):
    """Adds sequential IDs to a DataFrame for comparison with database tables."""
    df = df.copy()
    df["id"] = df.index + 1
    return df


def prepare_pandas_join_result(
    books_dataframe,
    category_dataframe,
    availability_dataframe
):
    """Reproduce the SQL JOIN using pandas.merge()."""

    merged_dataframe = pd.merge(
        books_dataframe,
        category_dataframe,
        left_on="category_id",
        right_on="id",
        how="left"
    )

    merged_dataframe = pd.merge(
        merged_dataframe,
        availability_dataframe,
        left_on="availability_id",
        right_on="id",
        how="left",
        suffixes=("", "_availability")
    )

    merged_dataframe["available"] = merged_dataframe[
        "availability"
    ].map({
        True: "Yes",
        False: "No",
        1: "Yes",
        0: "No"
    })

    merged_dataframe = merged_dataframe[
        [
            "title",
            "price_gbp",
            "price_inr",
            "rating",
            "category",
            "available"
        ]
    ]

    return merged_dataframe

def standardize_join_result(dataframe):
    """Standardize SQL and pandas JOIN results for comparison."""

    dataframe = dataframe.copy()

    dataframe = dataframe[
        [
            "title",
            "price_gbp",
            "price_inr",
            "rating",
            "category",
            "available"
        ]
    ]

    dataframe = dataframe.sort_values(
        by=[
            "category",
            "rating",
            "title"
        ]
    ).reset_index(drop=True)

    return dataframe

    return df

In [7]:
books_dataframe["category_id"] = pd.to_numeric(
    books_dataframe["category_id"],
    errors="raise"
).astype(int)

category_dataframe["id"] = pd.to_numeric(
    category_dataframe["id"],
    errors="raise"
).astype(int)

availability_dataframe["id"] = pd.to_numeric(
    availability_dataframe["id"],
    errors="raise"
).astype(int)



In [8]:


# 10 - SQL JOIN VS PANDAS MERGE
# --------------------------------------------------

print("\n10 --------------------------------------------------")

print("Checking SQL JOIN result against pandas.merge() result...")

# SQL JOIN result
query_op_all_join = pd.read_sql_query(
    all_joined_data_query,
    connection
)
# pandas.merge() result
pandas_merge_result = prepare_pandas_join_result(
    books_dataframe,
    category_dataframe,
    availability_dataframe
)
# Standardize both DataFrames
sql_join_result = standardize_join_result(
    query_op_all_join
)

pandas_merge_result = standardize_join_result(
    pandas_merge_result
)



print("\n10 --------------------------------------------------")
print("Checking SQL JOIN result against pandas.merge() result...")

print("\nSQL JOIN result:")
print(sql_join_result)

print("\nPandas merge() result:")
print(pandas_merge_result)

print(
    "\nAre SQL JOIN and pandas.merge() results equal?",
    sql_join_result.equals(pandas_merge_result)
)

print(
    "Is length of SQL JOIN result and pandas.merge() result equal?",
    len(sql_join_result) == len(pandas_merge_result)
)


10 --------------------------------------------------
Checking SQL JOIN result against pandas.merge() result...


ValueError: You are trying to merge on object and int64 columns for key 'availability_id'. If you wish to proceed you should use pd.concat

In [9]:
print(
    pd.read_sql_query(
        """
        SELECT
            availability_id,
            typeof(availability_id) AS sqlite_type
        FROM books
        LIMIT 10
        """,
        connection
    )
)

  availability_id sqlite_type
0            None        null
1            None        null
2            None        null
3            None        null
4            None        null
5            None        null
6            None        null
7            None        null
8            None        null
9            None        null
